# GRT / RMT — Kaggle · Colab 점검
현재 수정본을 `/kaggle/working/GRT` 또는 `/content/GRT`에 업로드·압축 해제하거나, 수정본을 반영한 저장소를 clone한 다음 실행하세요. 첫 셀의 `REPO_DIR`를 실제 저장소 루트로 바꾸면 됩니다. GPU 런타임을 권장합니다.
CPU 계약 테스트 → 두 모델의 2-step 학습 및 길이별 평가 → GRT 분석을 실행합니다. 표준 V/N/M 및 small 백본을 유지하고 평가 표본 수만 줄입니다. 이 점검은 과제 수렴이나 연구 결과를 입증하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, datetime
REPO_DIR = Path('/kaggle/working/GRT' if Path('/kaggle').exists() else '/content/GRT')
TASK = 'copy'  # copy / reverse / passkey
assert TASK in {'copy', 'reverse', 'passkey'}
assert (REPO_DIR / 'scripts/analyze.py').exists(), '현재 수정본을 올리고 REPO_DIR를 확인하세요.'
os.chdir(REPO_DIR)
def run(*arguments):
    subprocess.run([sys.executable, *map(str, arguments)], check=True, cwd=REPO_DIR)
run('-m', 'pip', 'install', '-e', '.[dev]')
import torch, yaml
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUT = REPO_DIR / 'runs' / ('notebook_' + STAMP)
OUT.mkdir(parents=True)


In [ ]:
# CPU fixtures, including dropout resume equivalence and fixed-batch overfit.
report = OUT / 'pytest_output.txt'
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--junitxml', str(OUT / 'pytest.xml')],
                        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
report.write_text(result.stdout, encoding='utf-8')
print(result.stdout)
assert result.returncode == 0, f'테스트 실패: {report}'


In [ ]:
# Separate smoke run; this override is persisted in each resolved config.
override = {
    'training': {'max_steps': 2, 'batch_size': 2, 'warmup_steps': 0},
    'data': {'validation_samples': 4, 'test_samples': 4, 'num_workers': 0},
    'evaluation': {'every_steps': 1, 'batch_size': 2, 'warmup': 1, 'iterations': 2},
    'checkpoint': {'every_steps': 1},
    'rtla': {'every_steps': 1, 'batch_size': 2},
    'wandb': {'enabled': False},
}
override_path = OUT / 'smoke_override.yaml'
override_path.write_text(yaml.safe_dump(override), encoding='utf-8')
for model_name in ['rmt', 'grt']:
    run('scripts/train.py', '--config', 'configs/base.yaml', f'configs/{model_name}.yaml',
        f'configs/{TASK}.yaml', override_path, '--output-dir', OUT / model_name)


In [ ]:
# Check standalone evaluate, resume, and analyze entry points.
for model_name in ['rmt', 'grt']:
    run('scripts/train.py', '--resume', OUT / model_name / 'last.pt')
    run('scripts/evaluate.py', '--checkpoint', OUT / model_name / 'best.pt')
extended = 30 if TASK == 'passkey' else 20
run('scripts/analyze.py', '--checkpoint', OUT / 'grt/best.pt', '--split', 'test', '--segments', extended)
for model_name in ['rmt', 'grt']:
    rows = json.loads((OUT / model_name / 'evaluation.json').read_text())['results']
    print(model_name, [{k: row[k] for k in ['T', 'loss', 'token_accuracy', 'exact_match']} for row in rows])
from IPython.display import display, Image
panels = sorted((OUT / 'grt/rtla').glob(f'*T{extended}_test*.png'))
display(Image(filename=str(panels[-1])))
print('결과 및 테스트 로그:', OUT)


In [ ]:
# Download this archive or attach its pytest_output.txt when reporting failures.
import shutil
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print('산출물 ZIP:', archive)
from IPython.display import FileLink
display(FileLink(os.path.relpath(archive, REPO_DIR)))
# Colab에서 바로 다운로드하려면 다음 두 줄의 주석을 해제하세요.
# from google.colab import files
# files.download(archive)
